# Steerling-8B: Concept Attribution Analysis

Decompose predicted logits into **known**, **discovered**, and **epsilon** contributions:

$$\text{logit}(y_t) = \sum_i C^{known}_{i,t} + \sum_j C^{unk}_{j,t} + \epsilon_t$$

**Requirements:** Interpretable Steerling model, GPU with >= 18 GB VRAM

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from steerling.inference.causal_diffusion import SteerlingGenerator

## Load Model

In [ ]:
# model_id = "guidelabs/steerling-8b"

# model = AutoModel.from_pretrained(model_id, trust_remote_code=True, torch_dtype=torch.bfloat16)
# tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
# generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")
# backbone = generator.model.model if hasattr(generator.model, "model") else generator.model
# device = generator.device

# print(generator)
# print(f"Interpretable: {generator.is_interpretable}")

In [ ]:
model_id = "guidelabs/steerling-8b"

generator = SteerlingGenerator.from_pretrained(
    model_id,
    device="cuda",
    dtype=torch.bfloat16,
)

tokenizer = generator.tokenizer
backbone = generator.model.model if hasattr(generator.model, "model") else generator.model
device = generator.device

print(generator)
print(f"Interpretable: {generator.is_interpretable}")
print(f"Interpretable backbone: {type(backbone).__name__}")
print(f"Has known head: {hasattr(backbone, 'known_head')}")
print(f"Has discovered head: {getattr(backbone, 'unknown_head', None) is not None}")

## Utilities

In [ ]:
concepts_path = Path("../assets/concepts/known_concepts.csv")

if concepts_path.exists():
    if concepts_path.suffix == ".parquet":
        concepts_df = pd.read_parquet(concepts_path)
    elif concepts_path.suffix == ".csv":
        concepts_df = pd.read_csv(concepts_path)
    else:
        raise ValueError(f"Unsupported concept file type: {concepts_path.suffix}")

    print(f"Loaded {len(concepts_df)} concepts from {concepts_path}")
    print("Columns:", concepts_df.columns.tolist())
else:
    concepts_df = pd.DataFrame(columns=["concept_idx", "concept_name", "top_100_tokens"])
    print("No concept file configured. Using fallback labels: Known #id & discovered #id")


def concept_label(concept_id: int, concept_type: str = "known") -> str:
    """Map a concept ID to a human-readable label."""
    if concept_type == "discovered":
        return f"Discovered #{concept_id}"

    row = concepts_df[concepts_df["concept_idx"] == concept_id]
    if len(row) == 0:
        return f"Known #{concept_id}"
    return row.iloc[0]["concept_name"]


def concept_words(concept_id: int, concept_type: str = "known") -> str:
    """Get concept words for a concept."""
    if concept_type == "discovered":
        return ""

    row = concepts_df[concepts_df["concept_idx"] == concept_id]
    if len(row) == 0:
        return ""

    words = row.iloc[0].get("top_100_tokens", "")
    if isinstance(words, list):
        return ", ".join(words[:8])
    return str(words)


def decode_token(token_id: int) -> str:
    """Decode a single token ID to string."""
    text = tokenizer.decode([token_id])
    if text.strip() == "":
        return repr(text)
    return text


def _endofchunk_id() -> int | None:
    """Get the end-of-chunk token ID."""
    eoc = getattr(tokenizer, "endofchunk_token_id", None)
    if eoc is not None:
        return int(eoc)

    eoc = getattr(generator.tokenizer, "endofchunk_token_id", None)
    if eoc is not None:
        return int(eoc)

    return None


def find_chunk_boundaries(token_ids: torch.Tensor) -> list[tuple[int, int]]:
    """Find chunk boundaries delimited by end-of-chunk tokens."""
    ids = token_ids.squeeze()
    eoc_id = _endofchunk_id()

    if eoc_id is None:
        return [(0, len(ids))]

    eoc_positions = (ids == eoc_id).nonzero(as_tuple=True)[0].tolist()

    chunks: list[tuple[int, int]] = []
    prev = 0
    for pos in eoc_positions:
        if pos > prev:
            chunks.append((prev, pos))
        prev = pos + 1

    if prev < len(ids):
        chunks.append((prev, len(ids)))

    return chunks


def show_chunks(token_ids: torch.Tensor, print_chunks: bool = True) -> list[tuple[int, int]]:
    """Show chunks delimited by end-of-chunk tokens."""
    ids = token_ids.squeeze()
    chunks = find_chunk_boundaries(token_ids)
    if print_chunks:
        print(f"Found {len(chunks)} chunks:\n")

    for i, (start, end) in enumerate(chunks):
        text = tokenizer.decode(ids[start:end].tolist())
        if print_chunks:
            print(f"  Chunk {i}: positions [{start}, {end}) ({end - start} tokens)")
            print(f"    {text!r}")

    return chunks

In [ ]:
def _compute_dense_contrib(
    head,
    weights: torch.Tensor,
    target_W: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute contributions when dense concept weights are available."""
    bsz, tlen, n_concepts = weights.shape
    indices = torch.arange(n_concepts, device=weights.device).view(1, 1, -1).expand(bsz, tlen, -1)
    embeddings = head._get_embedding_weight()[:n_concepts]
    dots = torch.einsum("cd,btd->btc", embeddings, target_W)
    contrib = weights * dots
    return indices, weights, contrib


def _compute_sparse_contrib(
    head,
    topk_indices: torch.Tensor,
    topk_logits: torch.Tensor,
    target_W: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute contributions when only top-k concept logits are available."""
    emb = head._get_embedding(topk_indices)
    dots = torch.einsum("btkd,btd->btk", emb, target_W)
    weights = torch.sigmoid(topk_logits.float())
    contrib = weights * dots
    return topk_indices, weights, contrib


@torch.no_grad()
def compute_output_to_concept(outputs, logits: torch.Tensor, backbone) -> dict:
    """Exact output->concept decomposition for predicted tokens."""
    target_token_ids = logits.argmax(dim=-1)
    target_logits = torch.gather(logits, -1, target_token_ids.unsqueeze(-1)).squeeze(-1)
    target_W = backbone.transformer.lm_head.weight[target_token_ids]

    if outputs.known_weights is not None:
        known_indices, known_weights, known_contrib = _compute_dense_contrib(
            backbone.known_head,
            outputs.known_weights,
            target_W,
        )
    elif outputs.known_topk_indices is not None and outputs.known_topk_logits is not None:
        known_indices, known_weights, known_contrib = _compute_sparse_contrib(
            backbone.known_head,
            outputs.known_topk_indices,
            outputs.known_topk_logits,
            target_W,
        )
    else:
        bsz, tlen = target_token_ids.shape
        known_indices = torch.zeros((bsz, tlen, 0), dtype=torch.long, device=target_token_ids.device)
        known_weights = torch.zeros((bsz, tlen, 0), device=target_token_ids.device)
        known_contrib = torch.zeros((bsz, tlen, 0), device=target_token_ids.device)

    if getattr(backbone, "unknown_head", None) is not None and outputs.unknown_weights is not None:
        unk_indices, unk_weights, unk_contrib = _compute_dense_contrib(
            backbone.unknown_head,
            outputs.unknown_weights,
            target_W,
        )
    elif (
        getattr(backbone, "unknown_head", None) is not None
        and outputs.unknown_topk_indices is not None
        and outputs.unknown_topk_logits is not None
    ):
        unk_indices, unk_weights, unk_contrib = _compute_sparse_contrib(
            backbone.unknown_head,
            outputs.unknown_topk_indices,
            outputs.unknown_topk_logits,
            target_W,
        )
    else:
        bsz, tlen = target_token_ids.shape
        unk_indices = torch.zeros((bsz, tlen, 0), dtype=torch.long, device=target_token_ids.device)
        unk_weights = torch.zeros((bsz, tlen, 0), device=target_token_ids.device)
        unk_contrib = torch.zeros((bsz, tlen, 0), device=target_token_ids.device)

    epsilon_contrib = target_logits - known_contrib.sum(dim=-1) - unk_contrib.sum(dim=-1)

    return {
        "target_token_ids": target_token_ids,
        "target_logits": target_logits,
        "known_indices": known_indices,
        "known_weights": known_weights,
        "known_contributions": known_contrib,
        "unk_indices": unk_indices,
        "unk_weights": unk_weights,
        "unk_contributions": unk_contrib,
        "epsilon_contribution": epsilon_contrib,
    }

In [ ]:
COLOR_MAP = {"known": "steelblue", "discovered": "darkorange", "epsilon": "gray"}


def _collect_unified_entries(attr: dict, pos: int, batch_idx: int = 0) -> list[dict]:
    entries: list[dict] = []

    for ki in range(attr["known_indices"].shape[-1]):
        cid = int(attr["known_indices"][batch_idx, pos, ki])
        entries.append(
            {
                "label": concept_label(cid, "known"),
                "concept_type": "known",
                "concept_id": cid,
                "contribution": float(attr["known_contributions"][batch_idx, pos, ki]),
            }
        )

    for ui in range(attr["unk_indices"].shape[-1]):
        cid = int(attr["unk_indices"][batch_idx, pos, ui])
        entries.append(
            {
                "label": f"Discovered #{cid}",
                "concept_type": "discovered",
                "concept_id": cid,
                "contribution": float(attr["unk_contributions"][batch_idx, pos, ui]),
            }
        )

    entries.append(
        {
            "label": "epsilon (residual)",
            "concept_type": "epsilon",
            "concept_id": -1,
            "contribution": float(attr["epsilon_contribution"][batch_idx, pos]),
        }
    )

    return entries


def plot_unified_contributions(
    entries: list[dict],
    title: str = "Concept Contributions",
    top_k: int = 20,
    plot_epsilon: bool = True,
):
    from matplotlib.patches import Patch

    concept_entries = [e for e in entries if e["concept_type"] != "epsilon"]
    eps_entries = [e for e in entries if e["concept_type"] == "epsilon"]

    concept_entries.sort(key=lambda e: abs(e["contribution"]), reverse=True)
    display = concept_entries[:top_k] + eps_entries if plot_epsilon else concept_entries[:top_k]

    # Compute epsilon percentage from the full decomposition, not just displayed bars.
    total_abs = sum(abs(e["contribution"]) for e in entries)
    epsilon_abs = sum(abs(e["contribution"]) for e in eps_entries)
    epsilon_pct = 100.0 * epsilon_abs / total_abs if total_abs > 0 else 0.0

    labels = [e["label"] for e in display]
    values = [e["contribution"] for e in display]
    colors = [COLOR_MAP[e["concept_type"]] for e in display]

    y = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(10, max(3, len(labels) * 0.35)))
    ax.barh(y, values, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_xlabel("Contribution to target logit")
    ax.set_title(f"{title} | epsilon={epsilon_pct:.1f}%", fontsize=11)

    if plot_epsilon:
        ax.legend(
            handles=[
                Patch(facecolor=COLOR_MAP["known"], label="Known"),
                Patch(facecolor=COLOR_MAP["discovered"], label="Discovered"),
                Patch(facecolor=COLOR_MAP["epsilon"], label="Epsilon"),
            ],
            loc="lower right",
            fontsize=8,
        )
    else:
        ax.legend(
            handles=[
                Patch(facecolor=COLOR_MAP["known"], label="Known"),
                Patch(facecolor=COLOR_MAP["discovered"], label="Discovered"),
            ],
        )

    plt.tight_layout()
    plt.show()

## Which tokens are tied to a concept?

In [ ]:
@torch.no_grad()
def concept_top_vocab_tokens(
    concept_id: int,
    concept_type: str = "known",
    mass_fraction: float = 0.10,
) -> list[tuple[str, float]]:
    """Return top vocabulary tokens favored by one concept embedding."""
    if concept_type == "known":
        v = backbone.known_head._get_embedding(torch.tensor([concept_id], device=device)).squeeze(0)
    else:
        if getattr(backbone, "unknown_head", None) is None:
            raise ValueError("This model does not have an unknown concept head.")
        v = backbone.unknown_head._get_embedding(torch.tensor([concept_id], device=device)).squeeze(0)

    W = backbone.transformer.lm_head.weight

    # Keep matmul dtypes/devices aligned, then cast result for analysis.
    v = v.to(device=W.device, dtype=W.dtype)
    scores = (W @ v).to(torch.float32)

    abs_scores = scores.abs()
    sorted_vals, sorted_idx = torch.sort(abs_scores, descending=True)

    total_mass = sorted_vals.sum().clamp(min=1e-12)
    cumsum = torch.cumsum(sorted_vals, dim=0)
    cutoff = (cumsum >= total_mass * mass_fraction).nonzero(as_tuple=True)[0]
    n_tokens = (cutoff[0].item() + 1) if len(cutoff) > 0 else len(sorted_vals)

    result: list[tuple[str, float]] = []
    for i in range(n_tokens):
        token_id = sorted_idx[i].item()
        score = scores[token_id].item()
        result.append((decode_token(token_id), score))

    return result


def show_concept_vocab(
    concept_id: int,
    concept_type: str = "discovered",
    mass_fraction: float = 0.10,
    max_display: int = 30,
) -> None:
    label = concept_label(concept_id, concept_type)
    tokens = concept_top_vocab_tokens(concept_id, concept_type, mass_fraction)

    print(f"\nConcept: {label} (id={concept_id}, {concept_type})")
    if concept_type == "discovered":
        words = concept_words(concept_id, concept_type)
        if words:
            print(f"  Lifted words: {words}")

    print(f"  Top vocab tokens ({mass_fraction * 100:.0f}% norm mass, {len(tokens)} tokens):")
    for tok, score in tokens[:max_display]:
        direction = "+" if score > 0 else "-"
        print(f"    {direction} {tok!r:>20s}  score={score:+.4f}")

    if len(tokens) > max_display:
        print(f"    ... and {len(tokens) - max_display} more")

In [ ]:
for cid in [0, 1, 5]:
    show_concept_vocab(cid, "known", mass_fraction=0.10, max_display=20)

if getattr(backbone, "unknown_head", None) is not None:
    for cid in [0, 1, 5]:
        show_concept_vocab(cid, "discovered", mass_fraction=0.10, max_display=20)

## Generate Text

In [ ]:
prompt = "AI technology will"

full_ids = generator.generate(prompt, gen_length=128, steps=128, temperature=0.4)

prompt_len = len(tokenizer.encode(prompt))
full_len = full_ids.shape[1]
gen_len = full_len - prompt_len

In [ ]:
with torch.no_grad():
    logits, outputs = backbone(full_ids, minimal_output=False)

print("\nFull generated text:")
print(tokenizer.decode(full_ids[0]))
print(
    "------------------------------------------------------------------------------------------------------------------------------"
)
print("\nFull sequence chunks:")
full_chunks = show_chunks(full_ids)
print(f"\nGenerated region: positions [{prompt_len}, {prompt_len + gen_len})")

## Output-to-Concept Attribution

Question: **Which concepts drove each predicted output token?**

For concept `i` at position `t` and predicted token `y_t`:

`contribution(i, y_t) = weight_i * (embedding_i dot W_{y_t})`

In [ ]:
o2c_attr = compute_output_to_concept(outputs, logits, backbone)

# First generated position
pos = prompt_len
pred_token = decode_token(int(o2c_attr["target_token_ids"][0, pos]))

# Mid-generation position
pos_mid = prompt_len + min(max(gen_len // 2, 0), gen_len - 1)
pred_token_mid = decode_token(int(o2c_attr["target_token_ids"][0, pos_mid]))

print(f"\nPosition {pos_mid}: predicting {pred_token_mid!r}")
entries_mid = _collect_unified_entries(o2c_attr, pos=pos_mid)
plot_unified_contributions(
    entries_mid,
    title=f"All contributions at pos {pos_mid} -> {pred_token_mid!r}",
    top_k=20,
    plot_epsilon=False,
)

## Chunk-to-Concept Attribution

In [ ]:
def _normalize_contributions(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    total = x.sum(dim=-1, keepdim=True)
    sign = torch.sign(total)
    sign = torch.where(sign == 0, torch.ones_like(sign), sign)
    denom = torch.clamp(torch.abs(total), min=eps)
    return x / (sign * denom)


def compute_chunk_concept_attribution(
    o2c_attr: dict,
    start: int,
    end: int,
    batch_idx: int = 0,
) -> dict:
    known_chunk = o2c_attr["known_contributions"][batch_idx, start:end, :]  # [T, K]
    unk_chunk = o2c_attr["unk_contributions"][batch_idx, start:end, :]  # [T, U]
    eps_chunk = o2c_attr["epsilon_contribution"][batch_idx, start:end]  # [T]

    n_tokens = end - start
    if n_tokens <= 0:
        raise ValueError("Chunk must contain at least one token.")

    known_indices = o2c_attr["known_indices"][batch_idx, start, :]
    unk_indices = o2c_attr["unk_indices"][batch_idx, start, :]

    joint = torch.cat(
        [
            known_chunk,
            unk_chunk,
            eps_chunk.unsqueeze(-1),
        ],
        dim=-1,
    )  # [T, K + U + 1]

    joint_norm = _normalize_contributions(joint)  # [T, K + U + 1]
    joint_mean = joint_norm.mean(dim=0)  # [K + U + 1]

    k = known_chunk.shape[-1]
    u = unk_chunk.shape[-1]

    known_scores = joint_mean[:k]
    unk_scores = joint_mean[k : k + u]
    epsilon_score = joint_mean[k + u]

    return {
        "start": start,
        "end": end,
        "known_indices": known_indices,
        "known_scores": known_scores,
        "unk_indices": unk_indices,
        "unk_scores": unk_scores,
        "epsilon_score": epsilon_score,
        "epsilon_raw_mean": eps_chunk.mean(),
        "epsilon_raw_abs_mean": eps_chunk.abs().mean(),
    }


def collect_chunk_contribution_entries(chunk_attr: dict) -> list[dict]:
    entries = []

    for i in range(len(chunk_attr["known_indices"])):
        cid = int(chunk_attr["known_indices"][i])
        entries.append(
            {
                "label": concept_label(cid, "known"),
                "concept_type": "known",
                "concept_id": cid,
                "contribution": float(chunk_attr["known_scores"][i]),
            }
        )

    for i in range(len(chunk_attr["unk_indices"])):
        cid = int(chunk_attr["unk_indices"][i])
        entries.append(
            {
                "label": f"Discovered #{cid}",
                "concept_type": "discovered",
                "concept_id": cid,
                "contribution": float(chunk_attr["unk_scores"][i]),
            }
        )

    entries.append(
        {
            "label": "epsilon (residual)",
            "concept_type": "epsilon",
            "concept_id": -1,
            "contribution": float(chunk_attr["epsilon_score"]),
        }
    )

    return entries


def plot_chunk_concept_attribution(
    chunk_attr: dict,
    title: str | None = None,
    top_k: int = 20,
    plot_epsilon: bool = True,
) -> None:
    entries = collect_chunk_contribution_entries(chunk_attr)
    if title is None:
        title = f"Chunk-level concept attribution [{chunk_attr['start']}, {chunk_attr['end']})"
    plot_unified_contributions(entries, title=title, top_k=top_k, plot_epsilon=plot_epsilon)

In [ ]:
# Lets examine the first chunk
chunk_start, chunk_end = full_chunks[0]

chunk_attr = compute_chunk_concept_attribution(
    o2c_attr,
    start=chunk_start,
    end=chunk_end,
)

plot_chunk_concept_attribution(
    chunk_attr,
    title=f"Chunk-level concept attribution [{chunk_start}, {chunk_end})",
    top_k=20,
    plot_epsilon=False,
)

In [ ]:
# Lets examine the second chunk
chunk_start, chunk_end = full_chunks[1]

chunk_attr = compute_chunk_concept_attribution(
    o2c_attr,
    start=chunk_start,
    end=chunk_end,
)

plot_chunk_concept_attribution(
    chunk_attr,
    title=f"Chunk-level concept attribution [{chunk_start}, {chunk_end})",
    top_k=20,
    plot_epsilon=False,
)